# Module 13: Agent Harness & Loop Engineering

**Goal:** Build agents that don't get stuck, crash, or over-run.

**What you'll learn:**
- Loop-until-dry with novelty gates
- Budget-aware loops with token tracking
- Durable journals for crash-proof resume
- Self-repair loops with retry logic

**Prerequisites:** Module 07 (agentic workflows basics)

---

## 0. Setup

In [ ]:
import hashlib
import json
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

print("✅ Setup complete")

---
## 1. Novelty Gate — Loop-Until-Dry

Keep running until K consecutive rounds produce nothing new.

In [ ]:
class NoveltyGate:
    def __init__(self, dry_threshold: int = 3):
        self.seen = set()
        self.dry_rounds = 0
        self.dry_threshold = dry_threshold

    def fingerprint(self, item):
        return hashlib.md5(str(item).encode()).hexdigest()

    def filter_new(self, items):
        new_items = [i for i in items if self.fingerprint(i) not in self.seen]
        if new_items:
            for item in new_items:
                self.seen.add(self.fingerprint(item))
            self.dry_rounds = 0
        else:
            self.dry_rounds += 1
        return new_items

    @property
    def is_dry(self):
        return self.dry_rounds >= self.dry_threshold

# Simulate: 5 new items → 3 new → 0 → 0 → 0 (should stop)
gate = NoveltyGate(dry_threshold=3)
rounds = [[1, 2, 3, 4, 5], [6, 7, 8], [], [], []]

for i, items in enumerate(rounds):
    fresh = gate.filter_new(items)
    print(f"Round {i+1}: {len(items)} items → {len(fresh)} new, dry={gate.dry_rounds}, is_dry={gate.is_dry}")
    if gate.is_dry:
        print(f"  → Stopped after round {i+1}")
        break

---
## 2. Budget-Aware Loop

Stop when token budget is exhausted. Inject remaining budget into agent context.

In [ ]:
@dataclass
class BudgetTracker:
    total_tokens: int
    spent_tokens: int = 0

    def record(self, input_tokens: int, output_tokens: int):
        self.spent_tokens += input_tokens + output_tokens

    @property
    def remaining(self):
        return max(0, self.total_tokens - self.spent_tokens)

    @property
    def is_exhausted(self):
        return self.remaining < 1000

# Simulate a loop consuming budget
budget = BudgetTracker(total_tokens=10_000)
iteration = 0

while not budget.is_exhausted:
    iteration += 1
    # Simulate: each iteration uses ~2000 tokens
    budget.record(input_tokens=1500, output_tokens=500)
    print(f"Iteration {iteration}: spent={budget.spent_tokens:,}, remaining={budget.remaining:,}")

print(f"\nStopped after {iteration} iterations (budget nearly exhausted)")

---
## 3. Durable Journal — Crash-Proof Resume

Write each completed step to an append-only log. On restart, replay to skip completed work.

In [ ]:
class Journal:
    def __init__(self, run_id: str, journal_dir: str = ".journals"):
        self.path = Path(journal_dir) / f"{run_id}.jsonl"
        self.path.parent.mkdir(exist_ok=True)
        self._completed = {}
        self._load()

    def _load(self):
        if self.path.exists():
            with open(self.path) as f:
                for line in f:
                    entry = json.loads(line)
                    self._completed[entry["step_id"]] = entry["result"]
            print(f"  Resumed: {len(self._completed)} steps already done")

    def step_id(self, name, *args):
        return hashlib.md5(f"{name}:{':'.join(str(a) for a in args)}".encode()).hexdigest()[:12]

    def is_done(self, sid):
        return sid in self._completed

    def execute(self, name, fn, *args):
        sid = self.step_id(name, *args)
        if self.is_done(sid):
            print(f"  ↩ Skipping '{name}' (already done)")
            return self._completed[sid]
        result = fn(*args)
        with open(self.path, "a") as f:
            f.write(json.dumps({"step_id": sid, "step_name": name, "result": result, "timestamp": datetime.now().isoformat()}) + "\n")
        self._completed[sid] = result
        print(f"  ✓ Completed '{name}'")
        return result

# Demo: run, crash, resume
journal = Journal(run_id="demo_run")
results = []
for topic in ["context engineering", "agent harness", "MCP"]:
    result = journal.execute("summarize", lambda t: f"Summary of {t}", topic)
    results.append(result)

print(f"\nResults: {results}")

---
## 4. Outcome Loop — iterate → grade → revise

The loops above stop on *mechanical* conditions (nothing new, budget gone). An **outcome loop** stops on a *quality* condition: you declare "done" as a gradeable rubric, and a grader with a **fresh context** scores each attempt and reports the gaps.

The grader never sees how the artifact was built, so it can't be talked into approving the worker's own reasoning. Feed back **only the unmet criteria** — replaying the whole rubric teaches the model nothing about what it missed.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Gap:
    criterion: str
    why: str

@dataclass
class Verdict:
    satisfied: bool
    gaps: list = field(default_factory=list)


def outcome_loop(task, rubric, worker, grader, max_iterations=5):
    """Iterate -> grade with a fresh context -> revise. Two stop conditions."""
    artifact, feedback = "", ""
    for i in range(1, max_iterations + 1):
        artifact = worker(task, rubric, artifact, feedback)
        verdict = grader(rubric, artifact)
        print(f"  [iteration {i}] " +
              ("satisfied" if verdict.satisfied else f"{len(verdict.gaps)} gap(s)"))
        for g in verdict.gaps:
            print(f"      x {g.criterion}: {g.why}")
        if verdict.satisfied:
            return artifact, "satisfied", i
        feedback = "\n".join(f"- {g.criterion}: {g.why}" for g in verdict.gaps)
    return artifact, "max_iterations_reached", max_iterations


# --- Scripted worker/grader so this runs with no API key -----------------
_round = {"n": 0}

def mock_worker(task, rubric, prior, feedback):
    _round["n"] += 1
    parts = ["# Quarterly Update"]
    if _round["n"] >= 2:
        parts += ["- Revenue up 12%", "- Churn flat", "- Two new regions"]
    if _round["n"] >= 3:
        parts.append("Summary: growth held, retention stable.")
    return "\n".join(parts)

def mock_grader(rubric, artifact):
    gaps = []
    if not artifact.startswith("#"):
        gaps.append(Gap("has a title", "no markdown heading"))
    bullets = [ln for ln in artifact.splitlines() if ln.startswith("- ")]
    if len(bullets) != 3:
        gaps.append(Gap("has exactly 3 bullets", f"found {len(bullets)}, expected 3"))
    if "Summary:" not in artifact:
        gaps.append(Gap("ends with a summary line", "no Summary line"))
    return Verdict(satisfied=not gaps, gaps=gaps)


RUBRIC = """- Starts with a markdown H1 title
- Contains exactly 3 bullet points
- Ends with a single line beginning "Summary:"
"""

artifact, result, iters = outcome_loop(
    "Write a short Q4 update for the exec team.", RUBRIC, mock_worker, mock_grader
)
print(f"\nResult: {result} after {iters} iteration(s)\n")
print(artifact)

---
## 5. API-Native Loop Controls

Two of the controls above no longer have to live in your code.

**`effort`** — the per-call depth dial. Lower it on worker/verification steps, raise it on the main loop:

```python
client.messages.create(
    model="claude-opus-5-5",
    max_tokens=64000,
    thinking={"type": "adaptive"},
    output_config={"effort": "high"},   # low | medium | high | xhigh | max
    tools=tools, messages=messages,
)
```

**`task_budget`** — a budget the *model* can see, tracked server-side, so it paces itself and finishes gracefully instead of being cut off:

```python
with client.beta.messages.stream(
    model="claude-opus-5-5",
    max_tokens=128000,
    betas=["task-budgets-2026-03-13"],
    output_config={"effort": "high",
                   "task_budget": {"type": "tokens", "total": 64000}},  # min 20,000
    tools=tools, messages=messages,
) as stream:
    response = stream.get_final_message()
```

- `max_tokens` is a hard ceiling the model is **unaware** of — hitting it truncates.
- `task_budget` is a target the model **sees** — it wraps up instead of being cut off.
- Prompt-injecting "you have N tokens left" also works, but it changes the prefix every turn and destroys your prompt cache.

**Context lifecycle:** long loops must also prune. Observation masking first (cheapest), then server-side context editing (`clear_tool_uses_20250919`) or compaction (`compact_20260112`). With compaction, append the **full `response.content`** back into `messages` — the compaction block lives there, and dropping it silently loses the compaction state.

---
## Summary

| Pattern | Purpose |
|---------|---------|
| **Novelty Gate** | Stop when no new items found (exhaustive search) |
| **Budget Tracker** | Stop when token/cost budget exhausted |
| **Durable Journal** | Crash-proof resume without re-executing completed steps |
| **Self-Repair** | Retry with error context fed back to agent |
| **Outcome Loop** | Stop when a graded rubric is satisfied |
| **`task_budget` / `effort`** | Let the model pace and scale itself |
| **Context editing / compaction** | Keep a long loop's context from rotting |

**The harness is what separates a demo from a production agent. Build it first.**

---
## 🧪 Exercises

1. **Novelty Gate Calibration**: Simulate rounds producing 5, 3, 0, 0, 0 items. With threshold=3, does it stop after round 3 or round 5? Why?

2. **Journal Resume**: Write a 5-step pipeline. After step 3, raise an exception. On restart, verify steps 1-3 are skipped.

3. **Budget Governor**: Run a loop with 50,000 token budget, ~5,000 tokens/iteration. Verify it stops before exceeding budget.

4. **Outcome Loop**: Re-run cell 4 with `max_iterations=2`. What `result` do you get, and which gap is still open? Now make the rubric criterion "has exactly 3 bullets" vague ("has some bullets") and explain why the loop becomes unpredictable.

5. **Budget Comparison**: Run the same task with (a) no budget, (b) budget injected into the prompt each turn, (c) API-native `task_budget`. Compare total tokens and cache-read hit rate.
